# Baseline SASRec (Random IDs)

**Part 2b of the Sequential Recommenders series**

Standard SASRec from Part 1: each game is a single integer, the model learns item embeddings from scratch.

**Prerequisites:** Run the RQVAE notebook first to generate checkpoints 1-2.


---
## 1. Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import gc
import ast
import gzip
import json
import time
import pickle
import subprocess
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict, NamedTuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from tqdm.notebook import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"

%matplotlib inline
sns.set_style('whitegrid')

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")
if device == 'cuda':
    torch.set_float32_matmul_precision('high')
    print(f"GPU: {torch.cuda.get_device_name()}")

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cuda
GPU: NVIDIA A10G


---
## Load Checkpoints

Load pre-computed embeddings, data, and semantic IDs from the RQVAE notebook.


In [2]:
# ============================================================
# Load Checkpoint 1 (run this instead of cells 2-6 to resume)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

item_embeddings = torch.load(f'{CHECKPOINT_DIR}/item_embeddings.pt', weights_only=True)
with open(f'{CHECKPOINT_DIR}/data_checkpoint1.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    games_df = _ckpt['games_df']
    user_sequences = _ckpt['user_sequences']
    sasrec_game_ids = _ckpt['sasrec_game_ids']
    del _ckpt

print(f'Checkpoint 1 loaded: item_embeddings {item_embeddings.shape}, '
      f'{len(games_df)} games, {len(user_sequences)} users')

Checkpoint 1 loaded: item_embeddings torch.Size([5232, 1024]), 5232 games, 56808 users


In [3]:
# ============================================================
# Load Checkpoint 2 (run this + checkpoint 1 load to skip RQVAE)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

with open(f'{CHECKPOINT_DIR}/data_checkpoint2.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    semantic_ids_3 = _ckpt['semantic_ids_3']
    semantic_ids_4 = _ckpt['semantic_ids_4']
    semantic_token_ids = _ckpt['semantic_token_ids']
    app_id_to_tokens = _ckpt['app_id_to_tokens']
    CODEBOOK_SIZE = _ckpt['CODEBOOK_SIZE']
    NUM_LEVELS = _ckpt['NUM_LEVELS']
    del _ckpt

print(f'Checkpoint 2 loaded: {len(app_id_to_tokens)} items, '
      f'{NUM_LEVELS} levels x {CODEBOOK_SIZE} codes')

Checkpoint 2 loaded: 5232 items, 4 levels x 256 codes


---
## 6. Baseline SASRec (Random IDs)

Standard SASRec from Part 1: each game is a single integer, the model learns item embeddings from scratch.

In [4]:
# ============================================================
# SASRec Model
# ============================================================

class SASRec(nn.Module):
    def __init__(self, item_num, max_seq_length=100, hidden_units=64,
                 num_blocks=2, num_heads=1, dropout_rate=0.2):
        super().__init__()
        self.item_num = item_num
        self.max_seq_length = max_seq_length
        self.hidden_units = hidden_units
        
        self.item_emb = nn.Embedding(item_num + 1, hidden_units, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_length + 1, hidden_units, padding_idx=0)
        self.emb_dropout = nn.Dropout(dropout_rate)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_units, nhead=num_heads,
            dim_feedforward=hidden_units, dropout=dropout_rate,
            activation='relu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_blocks)
        self.ln_f = nn.LayerNorm(hidden_units, eps=1e-8)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)
            if m.padding_idx is not None:
                with torch.no_grad(): m.weight[m.padding_idx].fill_(0)
    
    def forward(self, input_ids):
        B, T = input_ids.size()
        embs = self.item_emb(input_ids) * self.hidden_units**0.5
        positions = torch.arange(1, T+1, device=input_ids.device).unsqueeze(0).expand(B, -1)
        positions = positions * (input_ids != 0).long()
        hidden = self.emb_dropout(embs + self.pos_emb(positions))
        causal_mask = nn.Transformer.generate_square_subsequent_mask(T, device=input_ids.device)
        hidden = self.transformer(hidden, mask=causal_mask, is_causal=True)
        return self.ln_f(hidden)
    
    def predict(self, input_ids, candidate_ids):
        hidden = self.forward(input_ids)[:, -1, :]  # [B, H]
        cand_embs = self.item_emb(candidate_ids)     # [B, C, H]
        return torch.bmm(cand_embs, hidden.unsqueeze(-1)).squeeze(-1)
    
    def training_step(self, input_ids, pos_ids, neg_ids):
        hidden = self.forward(input_ids)
        pos_logits = (hidden * self.item_emb(pos_ids)).sum(-1)
        neg_logits = (hidden * self.item_emb(neg_ids)).sum(-1)
        return pos_logits, neg_logits


print('SASRec model defined')

SASRec model defined


In [5]:
# ============================================================
# Prepare sequences for SASRec
# ============================================================

# Map app_ids to integer indices (1-indexed, 0 = padding)
games_in_metadata = set(games_df['app_id'].tolist())
valid_app_ids = sorted(games_in_metadata & sasrec_game_ids)
item_to_id = {aid: idx + 1 for idx, aid in enumerate(valid_app_ids)}
id_to_item = {v: k for k, v in item_to_id.items()}

# Build user sequences (sorted by playtime descending as engagement proxy)
MAX_SEQ_LEN = 100
user_seqs = {}
for user, items in user_sequences.items():
    seq = []
    for item in sorted(items, key=lambda x: x['hours'], reverse=True):
        if item['item_id'] in item_to_id:
            seq.append(item_to_id[item['item_id']])
    if len(seq) >= 3:
        user_seqs[user] = seq

n_items = len(item_to_id)
n_users = len(user_seqs)
avg_len = np.mean([len(s) for s in user_seqs.values()])
print(f'Items: {n_items:,}, Users: {n_users:,}, Avg seq len: {avg_len:.1f}')


def make_sasrec_batch(user_seqs, users, max_item, max_seq_len=100):
    """Create SASRec training batch with negative sampling."""
    B = len(users)
    seq_t = torch.zeros(B, max_seq_len, dtype=torch.long)
    pos_t = torch.zeros(B, max_seq_len, dtype=torch.long)
    neg_t = torch.zeros(B, max_seq_len, dtype=torch.long)
    
    for i, user in enumerate(users):
        seq = user_seqs[user][:-2]  # Exclude last 2 for val/test
        if len(seq) > max_seq_len:
            seq = seq[-max_seq_len:]
        sl = len(seq)
        seq_t[i, -sl:] = torch.tensor(seq)
        
        full_seq = user_seqs[user]
        seen = set(full_seq)
        for j in range(sl):
            pos_t[i, -sl+j] = seq[j+1] if j < sl-1 else full_seq[len(seq)]
            neg = np.random.randint(1, max_item + 1)
            while neg in seen:
                neg = np.random.randint(1, max_item + 1)
            neg_t[i, -sl+j] = neg
    
    return seq_t, pos_t, neg_t


def evaluate_sasrec(model, user_seqs, mode='val', n_neg=100, device='cpu'):
    """Evaluate SASRec: target + 100 negatives, compute Hit@10, NDCG@10."""
    model.eval()
    ranks = []
    users = list(user_seqs.keys())
    
    for batch_start in tqdm(range(0, len(users), 256), desc=f'Eval ({mode})', leave=False):
        batch_users = users[batch_start:batch_start+256]
        batch_data = []
        
        for user in batch_users:
            seq = user_seqs[user]
            if mode == 'val':
                if len(seq) < 3: continue
                inp, tgt = seq[:-2], seq[-2]
            else:
                if len(seq) < 2: continue
                inp, tgt = seq[:-1], seq[-1]
            batch_data.append((user, inp, tgt))
        
        if not batch_data: continue
        
        max_len = min(max(len(d[1]) for d in batch_data), MAX_SEQ_LEN)
        input_t = torch.zeros(len(batch_data), max_len, dtype=torch.long, device=device)
        cands = torch.zeros(len(batch_data), n_neg + 1, dtype=torch.long, device=device)
        
        for i, (user, inp, tgt) in enumerate(batch_data):
            sl = min(len(inp), max_len)
            input_t[i, -sl:] = torch.tensor(inp[-sl:], device=device)
            cands[i, 0] = tgt
            seen = set(user_seqs[user])
            j = 1
            while j <= n_neg:
                neg = np.random.randint(1, n_items + 1)
                if neg not in seen:
                    cands[i, j] = neg
                    j += 1
        
        with torch.no_grad():
            scores = model.predict(input_t, cands)
            _, idx = scores.sort(dim=1, descending=True)
            r = (idx == 0).nonzero(as_tuple=True)[1].cpu().numpy() + 1
            ranks.extend(r.tolist())
    
    ranks = np.array(ranks)
    hr10 = (ranks <= 10).mean()
    ndcg10 = np.where(ranks <= 10, 1.0 / np.log2(ranks + 1), 0.0).mean()
    return {'hr@10': hr10, 'ndcg@10': ndcg10}


print('Training and evaluation functions defined')

Items: 5,232, Users: 56,692, Avg seq len: 34.9
Training and evaluation functions defined


In [6]:
# ============================================================
# Checkpoint 3: Save sequence data (after cell above)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

with open(f'{CHECKPOINT_DIR}/data_checkpoint3.pkl', 'wb') as f:
    pickle.dump({
        'user_seqs': user_seqs,
        'item_to_id': item_to_id,
        'id_to_item': id_to_item,
        'n_items': n_items,
    }, f)
print(f'Checkpoint 3 saved: {len(user_seqs)} users, {n_items} items')

Checkpoint 3 saved: 56692 users, 5232 items


In [7]:
# ============================================================
# Load Checkpoint 3 (run this + checkpoints 1-2 to skip to training)
# ============================================================
CHECKPOINT_DIR = 'checkpoints'

with open(f'{CHECKPOINT_DIR}/data_checkpoint3.pkl', 'rb') as f:
    _ckpt = pickle.load(f)
    user_seqs = _ckpt['user_seqs']
    item_to_id = _ckpt['item_to_id']
    id_to_item = _ckpt['id_to_item']
    n_items = _ckpt['n_items']
    del _ckpt

MAX_SEQ_LEN = 100
print(f'Checkpoint 3 loaded: {len(user_seqs)} users, {n_items} items')

Checkpoint 3 loaded: 56692 users, 5232 items


In [8]:
# ============================================================
# Train baseline SASRec
# ============================================================

sasrec = SASRec(item_num=n_items, max_seq_length=MAX_SEQ_LEN,
                hidden_units=64, num_blocks=2, num_heads=1, dropout_rate=0.2).to(device)
print(f'SASRec params: {sum(p.numel() for p in sasrec.parameters()):,}')

opt = torch.optim.AdamW(sasrec.parameters(), lr=1e-3, betas=(0.9, 0.98))
bce = nn.BCEWithLogitsLoss()
all_users = list(user_seqs.keys())
BATCH_SIZE_SR = 1024
NUM_EPOCHS_SR = 200

best_ndcg = 0
for epoch in tqdm(range(NUM_EPOCHS_SR), desc="SASRec Training"):
    sasrec.train()
    np.random.shuffle(all_users)
    epoch_loss = 0
    n_batches = 0

    for i in range(0, len(all_users), BATCH_SIZE_SR):
        batch_users = all_users[i:i+BATCH_SIZE_SR]
        seq_t, pos_t, neg_t = make_sasrec_batch(user_seqs, batch_users, n_items, MAX_SEQ_LEN)
        seq_t, pos_t, neg_t = seq_t.to(device), pos_t.to(device), neg_t.to(device)
        
        opt.zero_grad()
        pos_logits, neg_logits = sasrec.training_step(seq_t, pos_t, neg_t)
        
        mask = (pos_t != 0)
        if mask.any():
            loss = bce(pos_logits[mask], torch.ones_like(pos_logits[mask])) + \
                   bce(neg_logits[mask], torch.zeros_like(neg_logits[mask]))
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            n_batches += 1
    
    if (epoch + 1) % 20 == 0:
        val = evaluate_sasrec(sasrec, user_seqs, 'val', device=device)
        print(f'Epoch {epoch+1:3d} | loss: {epoch_loss/max(n_batches,1):.4f} | '
              f'val HR@10: {val["hr@10"]:.4f} | NDCG@10: {val["ndcg@10"]:.4f}')
        if val['ndcg@10'] > best_ndcg:
            best_ndcg = val['ndcg@10']
            torch.save(sasrec.state_dict(), 'sasrec_best.pt')

# Final test
sasrec.load_state_dict(torch.load('sasrec_best.pt', map_location=device, weights_only=True))
test_baseline = evaluate_sasrec(sasrec, user_seqs, 'test', device=device)
print(f'\nBaseline SASRec Test: HR@10={test_baseline["hr@10"]:.4f}, NDCG@10={test_baseline["ndcg@10"]:.4f}')

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


SASRec params: 391,936


SASRec Training:   0%|          | 0/200 [00:00<?, ?it/s]

Eval (val):   0%|          | 0/222 [00:00<?, ?it/s]

Epoch  20 | loss: 0.4334 | val HR@10: 0.8837 | NDCG@10: 0.6238


...

Epoch 180 | loss: 0.3720 | val HR@10: 0.9198 | NDCG@10: 0.6611


Eval (val):   0%|          | 0/222 [00:00<?, ?it/s]

Epoch 200 | loss: 0.3717 | val HR@10: 0.9205 | NDCG@10: 0.6602


Eval (test):   0%|          | 0/222 [00:00<?, ?it/s]


Baseline SASRec Test: HR@10=0.8892, NDCG@10=0.6399


In [9]:
# Final test
sasrec.load_state_dict(torch.load('sasrec_best.pt', map_location=device, weights_only=True))
test_baseline = evaluate_sasrec(sasrec, user_seqs, 'test', device=device,n_neg=5000)
print(f'\nBaseline SASRec Test: HR@10={test_baseline["hr@10"]:.4f}, NDCG@10={test_baseline["ndcg@10"]:.4f}')

Eval (test):   0%|          | 0/222 [00:00<?, ?it/s]


Baseline SASRec Test: HR@10=0.2143, NDCG@10=0.1240
